# Rotary Position Embedding (RoPE)
*Encoding token positions by rotating embedding vectors — the positional scheme in LLaMA, Mistral, Qwen, GLM.*

**Companion kernels:** `v0_naive.py` → `sm89_v1_pair_coalesced.py`


## What Is RoPE?

**In plain English:** RoPE encodes the *position* of each token by rotating its query/key vectors. A rotation doesn't change the vector's length, only its direction. The key insight: when you dot-product two rotated vectors, the result depends only on their *relative* positions — not their absolute positions. This means attention naturally learns position-relative patterns (e.g., "adjacent" vs "far away").

**Why not learned positional embeddings?** Learned embeddings can't generalize beyond the training context length. RoPE extrapolates to longer sequences because it encodes relative offsets algebraically.


In [ ]:
import math
print('ready')

## Worked Example: 2D Rotation ($d = 2$)

A 2D vector $[x_0, x_1]$ rotated by angle $\theta$ becomes:

$$\begin{bmatrix} x_0' \\ x_1' \end{bmatrix} = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix} \begin{bmatrix} x_0 \\ x_1 \end{bmatrix}$$

Take $x = [1.0, 0.0]$ (a unit vector pointing right).

**Position $m=0$, rotation angle $= 0 \cdot 1.0 = 0$ rad:**

$$x' = [\cos(0), \sin(0)] = [1.0,\ 0.0] \quad \text{(unchanged)}$$

**Position $m=1$, rotation angle $= 1 \cdot 1.0 = 1.0$ rad ($\approx 57°$):**

$$x' = [\cos(1.0),\ \sin(1.0)] = [0.540,\ 0.841]$$

### The Relative Position Property

$\text{RoPE}(x, m=0) \cdot \text{RoPE}(x, m=1) = [1.0, 0.0] \cdot [0.540, 0.841] = 0.540$

This dot product depends only on the **difference** $\Delta m = 1$, not on the absolute positions.

### The Norm is Preserved (Isometry)

$\|[0.540, 0.841]\| = \sqrt{0.540^2 + 0.841^2} = \sqrt{0.292 + 0.707} = 1.0$ ✅

Rotation can never change the vector's length — only its direction.


In [ ]:
def rope_2d(x, m, theta=1.0):
    angle = m * theta
    c, s = math.cos(angle), math.sin(angle)
    return [c * x[0] - s * x[1],
            s * x[0] + c * x[1]]

x = [1.0, 0.0]
for m in range(4):
    xr = rope_2d(x, m)
    norm = math.sqrt(xr[0]**2 + xr[1]**2)
    angle_deg = m * 1.0 * 180 / math.pi
    print(f"  m={m}:  angle={angle_deg:6.1f}°  "
          f"x' = [{xr[0]:+.4f}, {xr[1]:+.4f}]  "
          f"|x'| = {norm:.6f}")

print()
# Relative position property: dot(m=0, m=1) vs dot(m=2, m=3) should be equal
xr0 = rope_2d(x, 0);  xr1 = rope_2d(x, 1)
xr2 = rope_2d(x, 2);  xr3 = rope_2d(x, 3)
dot_01 = xr0[0]*xr1[0] + xr0[1]*xr1[1]
dot_23 = xr2[0]*xr3[0] + xr2[1]*xr3[1]
print(f"dot(pos=0, pos=1) = {dot_01:.6f}")
print(f"dot(pos=2, pos=3) = {dot_23:.6f}")
print(f"Same? {abs(dot_01 - dot_23) < 1e-9}  ← relative distance Δm=1 → same dot product")


## The Formula (General $d$-Dimensional Case)

Split the $d$-dimensional vector into $d/2$ pairs. For pair $k$ (dimensions $2k$ and $2k+1$):

$$\theta_k = \frac{1}{10000^{2k / d}}$$

$$\begin{bmatrix} x_{2k}' \\ x_{2k+1}' \end{bmatrix} = \begin{bmatrix} \cos(m \theta_k) & -\sin(m \theta_k) \\ \sin(m \theta_k) & \cos(m \theta_k) \end{bmatrix} \begin{bmatrix} x_{2k} \\ x_{2k+1} \end{bmatrix}$$

| Symbol | Meaning |
|--------|---------|
| $m$ | Token position (0, 1, 2, …) |
| $k$ | Dimension pair index (0, 1, …, $d/2 - 1$) |
| $\theta_k$ | Base frequency for pair $k$ |
| $10000$ | The "base" (hyperparameter, sometimes 500000 for long context) |
| $\cos(m\theta_k), \sin(m\theta_k)$ | Rotation coefficients for position $m$ |

### 🗣️ Say It Out Loud
> *"For each dimension pair k, we rotate the pair by angle m times theta-sub-k. Theta-sub-k equals 1 over 10000 to the power of 2k divided by d."*

**Tutor's gloss:** "Different pairs rotate at different speeds. Pair 0 has $\theta_0 = 1.0$ — it spins fast (full rotation in ~6 positions). Pair $d/2-1$ has $\theta \approx 0$ — it spins very slowly (full rotation in ~600k positions). Fast pairs encode local (adjacent) relationships; slow pairs encode long-range (paragraph-level) context."


In [ ]:
def rope_frequencies(d, base=10000):
    return [1.0 / (base ** (2*k / d)) for k in range(d // 2)]

def rope(x, m, d, base=10000):
    thetas = rope_frequencies(d, base)
    result = list(x)
    for k, theta in enumerate(thetas):
        angle = m * theta
        c, s = math.cos(angle), math.sin(angle)
        i, j = 2*k, 2*k+1
        result[i] = c * x[i] - s * x[j]
        result[j] = s * x[i] + c * x[j]
    return result

d = 8
thetas = rope_frequencies(d)
print(f"Frequencies θ_k for d={d}:")
for k, th in enumerate(thetas):
    period = 2 * math.pi / th if th > 0 else float('inf')
    print(f"  k={k}: θ={th:.5f}  full rotation every {period:.1f} tokens")

print()
x_test = [1.0] * d
for m in [0, 1, 100, 1000]:
    xr = rope(x_test, m, d)
    norm = math.sqrt(sum(v**2 for v in xr))
    print(f"  m={m:5d}:  norm={norm:.6f}  (always 1 — isometry ✓)")


## Why RoPE Works Better Than Learned Positional Embeddings

| Property | Learned PE | RoPE |
|----------|-----------|------|
| Generalizes beyond training length | ❌ | ✅ (extrapolates by design) |
| Encodes relative positions | ❌ (absolute only) | ✅ (by the rotation algebra) |
| Parameter count | $T_\text{max} \times d$ | 0 (no parameters) |
| Context length extension | Requires retraining | Adjust base ($10000 \to 500000$) |

Used by: LLaMA 2/3, Mistral, Qwen, GLM-4/5, Gemma, Phi, Falcon.
